In [1]:
import numpy as np
import networkx as nx
import torch

import sys
sys.path.append('../')
from models.staged import STAGED
from utils.graph_data_handler import GraphDataHandler
from utils.graph_constructor import GraphConstructor
from tests.test_graph_constructor import create_test_data
from tests.test_graph_constructor import visualize_all_cells_comparison, visualize_graph, visualize_cell_positions
from utils.visualization import visualize_attention_graph

data = create_test_data()

graph_constructor = GraphConstructor(
    genes=data['genes'],
    ligand_receptor_pairs=data['ligand_receptor_pairs'],
    receptor_gene_pairs=data['receptor_gene_pairs'],
    cell_type_assignments=data['cell_type_assignments'],
    prior_grns=data['prior_grns']
)

# Define time lags as specified by the user
delta_gl = 1  # Time lag for gene -> ligand
delta_lr = 5  # Time lag for ligand -> receptor
delta_rg = 3  # Time lag for receptor -> gene
delta_gg = 7  # Time lag for gene -> gene

time_point = 10

# Ensure time_point is large enough to handle the lags
max_lag = max(delta_gl, delta_lr, delta_rg, delta_gg)
if time_point < max_lag:
    time_point = max_lag
    print(f"Adjusted time_point to {time_point} to handle time lags")

model = STAGED(
    num_genes=len(data['genes']),
    hidden_dim=64,
    num_gat_layers=1,
    num_mlp_layers=3,
    dropout=0.1,
    delta_gl=delta_gl,  # Time lag for gene -> ligand
    delta_lr=delta_lr,  # Time lag for ligand -> receptor 
    delta_rg=delta_rg,  # Time lag for receptor -> gene
    delta_gg=delta_gg,  # Time lag for gene -> gene
    add_self_loops=False,
)

graph_handler = GraphDataHandler(model)

# Prepare cell graphs
cell_graphs = []
for cell_idx in range(data['n_cells']):
    # Construct and update graph for each cell
    base_graph = graph_constructor.construct_base_graph(cell_idx)
    updated_graph = graph_constructor.update_graph_with_neighbors(
        base_graph, cell_idx, data['cell_positions'], 
        time_point, distance_threshold=10.0
    )
    # Assign features
    pyg_graph = graph_constructor.assign_node_features(
        updated_graph, cell_idx, time_point, 
        data['gene_expression'],
        delta_gl, delta_lr, delta_rg, delta_gg
    )
    cell_graphs.append(pyg_graph)

/gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/.conda_envs/mioflow/lib/python3.10/site-packages/torch_geometric/typing.py:54: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: /lib64/libm.so.6: version `GLIBC_2.29' not found (required by /vast/palmer/pi/krishnaswamy_smita/xingzhi/.conda_envs/mioflow/lib/python3.10/site-packages/libpyg.so)
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/.conda_envs/mioflow/lib/python3.10/site-packages/torch_geometric/typing.py:110: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: /lib64/libm.so.6: version `GLIBC_2.29' not found (required by /vast/palmer/pi/krishnaswamy_smita/xingzhi/.conda_envs/mioflow/lib/python3.10/site-packages/libpyg.so)
  warnings.warn(f"An issue occurred while importing 'torch-sparse'. "


In [2]:
cell_graphs

[Data(x=[24, 1], edge_index=[2, 26], gene_node_indices=[6], node_names=[24], node_types=[24]),
 Data(x=[18, 1], edge_index=[2, 20], gene_node_indices=[6], node_names=[18], node_types=[18]),
 Data(x=[18, 1], edge_index=[2, 20], gene_node_indices=[6], node_names=[18], node_types=[18]),
 Data(x=[18, 1], edge_index=[2, 20], gene_node_indices=[6], node_names=[18], node_types=[18]),
 Data(x=[18, 1], edge_index=[2, 20], gene_node_indices=[6], node_names=[18], node_types=[18]),
 Data(x=[18, 1], edge_index=[2, 20], gene_node_indices=[6], node_names=[18], node_types=[18]),
 Data(x=[18, 1], edge_index=[2, 20], gene_node_indices=[6], node_names=[18], node_types=[18])]

In [3]:
batch_size = 5
predictions, attention_weights, node_ptr =graph_handler.process_cell_graphs(
    cell_graphs,
    num_genes=len(data['genes']),
    batch_size=batch_size
)

In [4]:
predictions.shape

torch.Size([7, 6])

In [5]:
len(attention_weights)

2

In [6]:
attention_weights[0].shape

torch.Size([2, 146])

In [7]:
attention_weights[1].shape

torch.Size([146, 1])

In [9]:
attn_split = graph_handler.split_attention_by_graphs(attention_weights, node_ptr)
print(len(attn_split))
for i in range(len(attn_split)):
    print(attn_split[i][0].shape)
    # print(attn_split[i][1].shape)


7
torch.Size([2, 26])
torch.Size([2, 20])
torch.Size([2, 20])
torch.Size([2, 20])
torch.Size([2, 20])
torch.Size([2, 20])
torch.Size([2, 20])
